In [ ]:
pip install ultralytics


In [ ]:

# Image Deblurring Script
# Reqried packaegs
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import argparse
import os
import warnings
warnings.filterwarnings('ignore')

from scipy.signal import fftconvolve
from skimage import img_as_float, io
from skimage.draw import disk
from skimage.color import rgb2gray


In [ ]:

# PSF  Builders

def gaussian_psf(sigma, size=None):
    """Gaussian blur kernel."""
    if size is None:
        size = int(6 * sigma + 1) | 1  # ensure odd
    k = size // 2
    y, x = np.mgrid[-k:k+1, -k:k+1]
    g = np.exp(-(x**2 + y**2) / (2 * sigma**2))
    return g / g.sum()


def motion_psf(length, angle_deg):
    """Linear motion blur kernel."""
    size = 2 * length + 1
    psf = np.zeros((size, size))
    cx = cy = size // 2
    angle = np.deg2rad(angle_deg)
    for i in range(-length // 2, length // 2 + 1):
        x = int(round(cx + i * np.cos(angle)))
        y = int(round(cy + i * np.sin(angle)))
        if 0 <= x < size and 0 <= y < size:
            psf[y, x] = 1
    s = psf.sum()
    return psf / s if s > 0 else psf


def defocus_psf(radius):
    """Disk/defocus blur kernel."""
    size = 2 * radius + 3
    psf = np.zeros((size, size))
    rr, cc = disk((size // 2, size // 2), radius, shape=psf.shape)
    psf[rr, cc] = 1
    s = psf.sum()
    return psf / s if s > 0 else psf

In [ ]:

# Wiener Filter (Frequency Domain)


def wiener_filter(blurred, psf, K=0.01):
    def _channel(b, psf, K):
        H = np.fft.rfft2(psf, s=b.shape)
        B = np.fft.rfft2(b)
        W = np.conj(H) / (np.abs(H)**2 + K)
        restored = np.fft.irfft2(W * B, s=b.shape)
        return np.clip(restored, 0, 1)

    if blurred.ndim == 3:
        return np.stack([_channel(blurred[:, :, c], psf, K) for c in range(blurred.shape[2])], axis=2)
    return _channel(blurred, psf, K)

In [ ]:

# Richardson-Lucy Deconvolution (Iterative)
def richardson_lucy(blurred, psf, iterations=30):
    psf_flip = psf[::-1, ::-1]

    def _channel(b, psf, psf_flip, iters):
        u = np.full_like(b, 0.5)
        for _ in range(iters):
            conv = fftconvolve(u, psf, mode='same')
            conv = np.where(conv < 1e-9, 1e-9, conv)
            u *= fftconvolve(b / conv, psf_flip, mode='same')
            np.clip(u, 0, 1, out=u)
        return u
    if blurred.ndim == 3:
        return np.stack([_channel(blurred[:, :, c], psf, psf_flip, iterations) for c in range(blurred.shape[2])], axis=2)
    return _channel(blurred, psf, psf_flip, iterations)

In [ ]:

def compute_psnr(original, restored):
    from skimage.metrics import peak_signal_noise_ratio
    return peak_signal_noise_ratio(original, restored, data_range=1.0)
def compute_ssim(original, restored):
    from skimage.metrics import structural_similarity
    kw = dict(data_range=1.0)
    if original.ndim == 3:
        kw['channel_axis'] = 2
    return structural_similarity(original, restored, **kw)

In [ ]:
#To compare and save the image with both the module
def save_comparison(original, blurred, wiener_out, rl_out, output_path, metrics_dict=None):
    cmap = 'gray' if original.ndim == 2 else None
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    images = [original, blurred, wiener_out, rl_out]
    titles = ['Original', 'Blurred', 'Wiener Filter', 'Richardson-Lucy']
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(img, cmap=cmap, vmin=0, vmax=1)
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.axis('off')
    if metrics_dict:
        fig.text(0.5, -0.02,
            f"Wiener  — PSNR: {metrics_dict['wiener_psnr']:.2f} dB | SSIM: {metrics_dict['wiener_ssim']:.4f}    "
            f"RL  — PSNR: {metrics_dict['rl_psnr']:.2f} dB | SSIM: {metrics_dict['rl_ssim']:.4f}",
            ha='center', fontsize=12, color='#333333')
    plt.suptitle('Image Deblurring Comparison', fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(output_path, bbox_inches='tight', dpi=150)
    plt.close()
    print(f"Saved comparison → {output_path}")

In [ ]:
import os
import glob
import argparse
def main():
    parser = argparse.ArgumentParser(description='Deblur a folder of images using Wiener or Richardson-Lucy.')
    # Changed --image to --indir
    parser.add_argument('--indir',    required=True,          help='Path to input image directory')
    parser.add_argument('--blur',     default='gaussian',     choices=['gaussian', 'motion', 'defocus'],
                        help='Blur type to simulate (ignored if --no-simulate)')
    parser.add_argument('--sigma',    type=float, default=2,  help='Sigma for Gaussian blur')
    parser.add_argument('--length',   type=int,   default=20, help='Length for motion blur')
    parser.add_argument('--angle',    type=float, default=0,  help='Angle (degrees) for motion blur')
    parser.add_argument('--radius',   type=int,   default=10, help='Radius for defocus blur')
    parser.add_argument('--K',        type=float, default=0.01, help='Wiener regularization param')
    parser.add_argument('--iters',    type=int,   default=30, help='Richardson-Lucy iterations')
    parser.add_argument('--no-simulate', action='store_true',
                        help='Treat input images as already blurred')
    parser.add_argument('--outdir',   default='output_results', help='Output directory')
    args = parser.parse_args()

    os.makedirs(args.outdir, exist_ok=True)

    # 1. Gather all image files
    extensions = ['*.png', '*.jpg', '*.jpeg', '*.bmp', '*.tiff']
    image_paths = []
    for ext in extensions:
        image_paths.extend(glob.glob(os.path.join(args.indir, ext)))
    
    if not image_paths:
        print(f"No images found in {args.indir}")
        return

    # 2. Pre-build PSF (since it's the same for all images in the batch)
    if args.blur == 'gaussian':
        psf = gaussian_psf(args.sigma)
        blur_desc = f"Gaussian σ={args.sigma}"
    elif args.blur == 'motion':
        psf = motion_psf(args.length, args.angle)
        blur_desc = f"Motion {args.length}px @ {args.angle}°"
    else:
        psf = defocus_psf(args.radius)
        blur_desc = f"Defocus r={args.radius}"
    for img_path in image_paths:
        filename = os.path.basename(img_path)
        name_only = os.path.splitext(filename)[0]
        image = img_as_float(io.imread(img_path))
        h, w = image.shape[:2]
        image = image[:h - h % 2, :w - w % 2]
        if args.no_simulate:
            blurred = image
            original = None
        else:
            rng = np.random.default_rng(42)
            if image.ndim == 3:
                blurred = np.stack([fftconvolve(image[:, :, c], psf, mode='same') for c in range(image.shape[2])], axis=2)
            else:
                blurred = fftconvolve(image, psf, mode='same')
            blurred = np.clip(blurred + rng.normal(0, 0.01, blurred.shape), 0, 1)
            original = image
        wiener_out = wiener_filter(blurred, psf, K=args.K)
        rl_out = richardson_lucy(blurred, psf, iterations=args.iters)
        io.imsave(os.path.join(args.outdir, f'{name_only}_wiener.png'), (wiener_out * 255).astype(np.uint8))
        io.imsave(os.path.join(args.outdir, f'{name_only}_rl.png'),     (rl_out * 255).astype(np.uint8))
        metrics_dict = None
        if original is not None:
            metrics_dict = {
                'wiener_psnr': compute_psnr(original, wiener_out),
                'wiener_ssim': compute_ssim(original, wiener_out),
                'rl_psnr':     compute_psnr(original, rl_out),
                'rl_ssim':     compute_ssim(original, rl_out),
            }
        ref_for_plot = original if original is not None else blurred
        save_comparison(ref_for_plot, blurred, wiener_out, rl_out,
                        os.path.join(args.outdir, f'{name_only}_comparison.png'), metrics_dict)

    print(f"\n✅ Done! All results saved to: {os.path.abspath(args.outdir)}")
if __name__ == '__main__':
    import sys
    # Example setup for folder processing
    sys.argv = [
        'deblur_batch.py',
        '--indir', r"C:\Users\vcsd2\Downloads\gopro_deblur\blur\images",  # Path to your folder
        '--blur', 'gaussian',
        '--sigma', '2',
        '--iters', '30',
        '--outdir', './deblurred_results' 
    ]
    main()

In [ ]:
import os
from skimage import img_as_float, io
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import fftconvolve
SAMPLE_IMG = r"C:\Users\vcsd2\Downloads\gopro_deblur\blur\images\001326.png"  # adjust path
image = img_as_float(io.imread(SAMPLE_IMG))
h, w = image.shape[:2]
image = image[:h - h%2, :w - w%2]

# Build PSFs for different blur types and intensities
configs = [
    ('Gaussian σ=1', gaussian_psf(1)),
    ('Gaussian σ=3', gaussian_psf(3)),
    ('Motion 10px 0°', motion_psf(10, 0)),
    ('Motion 20px 45°', motion_psf(20, 45)),
    ('Defocus r=5', defocus_psf(5)),
    ('Defocus r=10', defocus_psf(10)),
]
rng = np.random.default_rng(42)
fig, axes = plt.subplots(len(configs), 3, figsize=(15, 4*len(configs)))
for row, (label, psf) in enumerate(configs):
    if image.ndim == 3:
        blurred = np.stack([fftconvolve(image[:,:,c], psf, mode='same') for c in range(3)], axis=2)
    else:
        blurred = fftconvolve(image, psf, mode='same')
    blurred = np.clip(blurred + rng.normal(0, 0.01, blurred.shape), 0, 1)
    wiener_out = wiener_filter(blurred, psf)
    rl_out = richardson_lucy(blurred, psf, iterations=30)
    psnr_w = compute_psnr(image, wiener_out)
    psnr_r = compute_psnr(image, rl_out)
    ssim_w = compute_ssim(image, wiener_out)
    ssim_r = compute_ssim(image, rl_out)
    cmap = None if image.ndim == 3 else 'gray'
    axes[row,0].imshow(blurred, cmap=cmap, vmin=0, vmax=1)
    axes[row,0].set_title(f'Blurred: {label}', fontweight='bold')
    axes[row,0].axis('off')
    axes[row,1].imshow(wiener_out, cmap=cmap, vmin=0, vmax=1)
    axes[row,1].set_title(f'Wiener  PSNR:{psnr_w:.1f} SSIM:{ssim_w:.3f}', fontweight='bold')
    axes[row,1].axis('off')
    axes[row,2].imshow(rl_out, cmap=cmap, vmin=0, vmax=1)
    axes[row,2].set_title(f'RL  PSNR:{psnr_r:.1f} SSIM:{ssim_r:.3f}', fontweight='bold')
    axes[row,2].axis('off')
plt.suptitle('Task 2 – Blur Type & Intensity Comparison', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('task2_multi_blur_comparison.png', bbox_inches='tight', dpi=120)
plt.show()
print('Saved task2_multi_blur_comparison.png')

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time
from ultralytics import YOLO
import os
model = YOLO("yolov8n.pt")
image_path = "deblurred_wiener.png"  # change this
if not os.path.exists(image_path):
    print(f"Error: Image file not found at {image_path}. Please ensure the previous cell successfully created it or provide a correct path.")
    raise FileNotFoundError(f"Image file not found: {image_path}")
original = cv2.imread(image_path)
if original is None:
    print(f"Error: OpenCV could not load the image from {image_path}. It might be corrupted or an invalid format.")
    raise ValueError(f"Failed to load image with OpenCV: {image_path}")
original = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)
def apply_blur(img):
    return cv2.GaussianBlur(img, (15, 15), 0)
blurred = apply_blur(original)
def deblur_image(img):
    kernel = np.array([[0, -1, 0],
                       [-1, 5,-1],
                       [0, -1, 0]])
    return cv2.filter2D(img, -1, kernel)

deblurred = deblur_image(blurred)
def run_detection(img):
    start = time.time()
    results = model(img)
    latency = time.time() - start
    boxes = results[0].boxes.xyxy.cpu().numpy()
    scores = results[0].boxes.conf.cpu().numpy()
    classes = results[0].boxes.cls.cpu().numpy()
    return boxes, scores, classes, results, latency
boxes_blur, scores_blur, cls_blur, res_blur, time_blur = run_detection(blurred)
boxes_deblur, scores_deblur, cls_deblur, res_deblur, time_deblur = run_detection(deblurred)
def draw_boxes(img, boxes, scores):
    img_copy = img.copy()
    for box, score in zip(boxes, scores):
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(img_copy, (x1, y1), (x2, y2), (0,255,0), 2)
        cv2.putText(img_copy, f"{score:.2f}", (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,0,0), 1)
    return img_copy
img_blur_boxes = draw_boxes(blurred, boxes_blur, scores_blur)
img_deblur_boxes = draw_boxes(deblurred, boxes_deblur, scores_deblur)
def compute_metrics(scores):
    if len(scores) == 0:
        return 0
    return np.mean(scores)

avg_conf_blur = compute_metrics(scores_blur)
avg_conf_deblur = compute_metrics(scores_deblur)
plt.figure(figsize=(12,6))
plt.subplot(1,2,1)
plt.imshow(img_blur_boxes)
plt.title(f"Blurred\nDetections: {len(scores_blur)} | Conf: {avg_conf_blur:.2f}")
plt.axis("off")
plt.subplot(1,2,2)
plt.imshow(img_deblur_boxes)
plt.title(f"Deblurred\nDetections: {len(scores_deblur)} | Conf: {avg_conf_deblur:.2f}")
plt.axis("off")
plt.show()
print("===== RESULTS =====")
print(f"Blurred:   {len(scores_blur)} objects | Avg Confidence: {avg_conf_blur:.2f} | Time: {time_blur:.3f}s")
print(f"Deblurred: {len(scores_deblur)} objects | Avg Confidence: {avg_conf_deblur:.2f} | Time: {time_deblur:.3f}s")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time
import os
import glob
import pandas as pd 
from ultralytics import YOLO
model = YOLO("yolov8n.pt")
input_dir = "./deblurred_results" 
output_dir = "./detection_results"
os.makedirs(output_dir, exist_ok=True)
def run_detection(img):
    start = time.time()
    results = model(img, verbose=False) # verbose=False keeps the console clean
    latency = time.time() - start
    boxes = results[0].boxes.xyxy.cpu().numpy()
    scores = results[0].boxes.conf.cpu().numpy()
    return boxes, scores, latency
def draw_boxes(img, boxes, scores):
    img_copy = img.copy()
    for box, score in zip(boxes, scores):
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(img_copy, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(img_copy, f"{score:.2f}", (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    return img_copy
wiener_images = glob.glob(os.path.join(input_dir, "*_wiener.png"))
if not wiener_images:
    print(f"No Wiener deblurred images found in {input_dir}")
else:
    print(f"Found {len(wiener_images)} images. Starting detection...")
all_stats = []
for img_path in wiener_images:
    filename = os.path.basename(img_path)
    name_only = filename.replace("_wiener.png", "")
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    boxes, scores, latency = run_detection(img_rgb)
    avg_conf = np.mean(scores) if len(scores) > 0 else 0
    num_objects = len(scores)
    detection_img = draw_boxes(img_rgb, boxes, scores)
    detection_img_bgr = cv2.cvtColor(detection_img, cv2.COLOR_RGB2BGR)
    cv2.imwrite(os.path.join(output_dir, f"{name_only}_detected.jpg"), detection_img_bgr)

    all_stats.append({
        "filename": filename,
        "objects_found": num_objects,
        "avg_confidence": round(float(avg_conf), 4),
        "latency_sec": round(latency, 4)
    })
    print(f"Processed {filename}: Found {num_objects} objects.")
df = pd.DataFrame(all_stats)
csv_path = os.path.join(output_dir, "detection_summary.csv")
df.to_csv(csv_path, index=False)

print("\n===== BATCH DETECTION COMPLETE =====")
print(f"Summary saved to: {csv_path}")
print(f"Images saved to: {os.path.abspath(output_dir)}")

In [ ]:
import os
from ultralytics import YOLO
model = YOLO('yolov8n.pt') 

source_folder = r"C:\Users\vcsd2\Downloads\gopro_deblur\blur\images" # Folder containing your images
output_log = 'detection_counts_blur.csv'       # File to store the results
results = model.predict(source=source_folder, conf=0.25, save=False, stream=True)

with open(output_log, 'w') as f:

    f.write("filename,object_count\n")
    
    print(f"Processing images in {source_folder}...")
    
    for result in results:
        filename = os.path.basename(result.path)
        count = len(result.boxes)
        f.write(f"{filename},{count}\n")
        print(f"Logged: {filename} - {count} objects")
print(f"--- Finished! Results saved to {output_log} ---")

In [ ]:
#YOLO module on the rl stored images
import os
import glob
from ultralytics import YOLO

model = YOLO('yolov8n.pt') 
source_folder = r'C:\Users\vcsd2\deblurred_results'  # Folder containing your images
output_log = 'filtered_detection_counts2.csv'
valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
image_list = [
    os.path.join(source_folder, f) for f in os.listdir(source_folder)
    if f.lower().endswith(valid_extensions) and '_wiener' in f.lower()
]

if not image_list:
    print("No images found matching the '_rl' criteria.")
else:
    print(f"Found {len(image_list)} matching images. Starting detection...")

    # 4. Run inference and log results
    with open(output_log, 'w') as f:
        f.write("filename,object_count\n")
        
        # We pass the list of filtered paths directly to the model
        results = model.predict(source=image_list, conf=0.25, save=False, stream=True)
        
        for result in results:
            filename = os.path.basename(result.path)
            count = len(result.boxes)
            
            f.write(f"{filename},{count}\n")
            print(f"Processed: {filename} | Detections: {count}")

    print(f"--- Task Complete! Log saved to {output_log} ---")

In [ ]:
import os
import glob
from ultralytics import YOLO
model = YOLO('yolov8n.pt') 

source_folder = r'C:\Users\vcsd2\deblurred_results' 
output_log = 'filtered_detection_counts.csv'
valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
image_list = [
    os.path.join(source_folder, f) for f in os.listdir(source_folder)
    if f.lower().endswith(valid_extensions) and '_rl' in f.lower()
]

if not image_list:
    print("No images found matching the '_rl' criteria.")
else:
    print(f"Found {len(image_list)} matching images. Starting detection...")

    # 4. Run inference and log results
    with open(output_log, 'w') as f:
        f.write("filename,object_count\n")
        
        # We pass the list of filtered paths directly to the model
        results = model.predict(source=image_list, conf=0.25, save=False, stream=True)
        
        for result in results:
            filename = os.path.basename(result.path)
            count = len(result.boxes)
            
            f.write(f"{filename},{count}\n")
            print(f"Processed: {filename} | Detections: {count}")

    print(f"--- Task Complete! Log saved to {output_log} ---")

In [ ]:
import os, shutil, random, yaml, glob
from ultralytics import YOLO

SOURCE_DIR = "./deblurred_results"    # Task 2 images are path
DATASET_ROOT = "./deblur_dataset"      # New training directory path
MODEL_VARIANT = "yolov8n.pt"           
TRAIN_SPLIT = 0.7                      # 70% train, 15% val, 15% test
subdirs = [
    'images/train', 'images/val', 'images/test',
    'labels/train', 'labels/val', 'labels/test'
]
for sd in subdirs:
    os.makedirs(os.path.join(DATASET_ROOT, sd), exist_ok=True)

images = [f for f in os.listdir(SOURCE_DIR) if f.endswith('_wiener.png')]

if not images:
    print(f"Error: No images found in {SOURCE_DIR}. Check your Task 2 output.")
    exit()

random.seed(42) # For reproducibility
random.shuffle(images)
images.sort()
split_train = int(len(images) * TRAIN_SPLIT)
split_val   = int(len(images) * (TRAIN_SPLIT + 0.15))
train_list = images[:split_train]
val_list   = images[split_train:split_val]
test_list  = images[split_val:]
print(f'Split → Train:{len(train_list)}  Val:{len(val_list)}  Test:{len(test_list)}')
labeler_model = YOLO(MODEL_VARIANT)

def build_yolo_set(file_list, stage):
    print(f"Organizing {stage} set ({len(file_list)} images)...")
    for fname in file_list:
        img_src = os.path.join(SOURCE_DIR, fname)
        img_dst = os.path.join(DATASET_ROOT, 'images', stage, fname)
        shutil.copy(img_src, img_dst)
        results = labeler_model(img_src, verbose=False)
        label_fname = os.path.splitext(fname)[0] + ".txt"
        label_path = os.path.join(DATASET_ROOT, 'labels', stage, label_fname)
        results[0].save_txt(label_path)

build_yolo_set(train_list, 'train')
build_yolo_set(val_list, 'val')
build_yolo_set(test_list, 'test')

dataset_info = {
    'path': os.path.abspath(DATASET_ROOT),
    'train': 'images/train',
    'val': 'images/val',
    'names': labeler_model.names # Use standard COCO names from the model
}

yaml_path = os.path.join(DATASET_ROOT, 'dataset.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_info, f, default_flow_style=False)

print(f"Dataset prepared and YAML created at: {yaml_path}")


print("\n Starting Fine-Tuning...")
train_model = YOLO(MODEL_VARIANT)
results = train_model.train(
    data=yaml_path,
    epochs=30,               # Adjust based on your needs
    imgsz=640,
    batch=8,
    name="wiener_deblur_detector",
    project="runs/detect",
    exist_ok=True,
    augment=True,            # Built-in: Mosaic, Flips, etc.
    lr0=0.01,                # Initial learning rate
    patience=10              # Early stopping if no improvement
)

print("\n Training Complete! Check 'runs/detect/wiener_deblur_detector' for results.")

In [ ]:

dataset_yaml = "deblur_dataset/dataset.yaml" 
original_model = YOLO("yolov8n.pt")
finetuned_model = YOLO(r"C:\Users\vcsd2\runs\detect\runs\detect\wiener_deblur_detector\weights\best.pt")

def get_stats(model, name, data_path):
    print(f"📊 Validating {name}...")
    # we use 'split=val' to check against your validation set
    results = model.val(data=data_path, split='val', verbose=False)
    
    return {
        "Domain": name,
        "mAP@50": round(results.results_dict['metrics/mAP50(B)'], 4),
        "Precision": round(results.results_dict['metrics/precision(B)'], 4),
        "Recall": round(results.results_dict['metrics/recall(B)'], 4),
        "Fitness": round(results.fitness, 4)
    }
results_list = []
results_list.append(get_stats(original_model, "Standard YOLO (on Deblurred)", dataset_yaml))
results_list.append(get_stats(finetuned_model, "Fine-tuned YOLO (on Deblurred)", dataset_yaml))
df = pd.DataFrame(results_list)
print("\n" + "="*60)
print("  TASK 5: FINAL PERFORMANCE COMPARISON")
print("="*60)
print(df.to_string(index=False))
print("="*60)
with open("performance_report.txt", "w") as f:
    f.write("FINAL PERFORMANCE COMPARISON REPORT\n")
    f.write("="*40 + "\n")
    f.write(df.to_string(index=False))
SHARP_DIR = r"C:\Users\vcsd2\Downloads\gopro_deblur\sharp\images"
BLUR_DIR  = r"C:\Users\vcsd2\Downloads\gopro_deblur\blur\images"
DEBLUR_DIR = "./deblurred_results"
sharp_model = YOLO('yolov8n.pt')
sample_names = [os.path.basename(p) for p in glob.glob(os.path.join(SHARP_DIR, '*.png'))[:50]]
rows = []
for fname in sample_names:
    sp = os.path.join(SHARP_DIR, fname)
    bp = os.path.join(BLUR_DIR, fname)
    dp = os.path.join(DEBLUR_DIR, fname.replace('.png', '_wiener.png'))
    for path, domain in [(bp,'blurred'),(dp,'deblurred'),(sp,'sharp')]:
        if not os.path.exists(path): continue
        img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        res = sharp_model(img, verbose=False)[0]
        conf = float(res.boxes.conf.mean()) if len(res.boxes.conf) else 0
        rows.append({'domain': domain, 'avg_conf': conf, 'n_det': len(res.boxes)})
df3 = pd.DataFrame(rows)
summary = df3.groupby('domain')[['avg_conf','n_det']].mean().round(4)
print('\n===== THREE-WAY COMPARISON: Blurred / Deblurred / Sharp =====')
print(summary.to_string())
fig, axes = plt.subplots(1, 2, figsize=(12,4))
summary['avg_conf'].plot(kind='bar', ax=axes[0], color=['steelblue','orange','green'])
axes[0].set_title('Avg Confidence by Domain'); axes[0].set_ylabel('Confidence')
axes[0].tick_params(axis='x', rotation=0)
summary['n_det'].plot(kind='bar', ax=axes[1], color=['steelblue','orange','green'])
axes[1].set_title('Avg Detections by Domain'); axes[1].set_ylabel('Detections')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig('task5_three_way_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved task5_three_way_comparison.png')
with open('performance_report.txt', 'a') as f:
    f.write('\n\nTHREE-WAY COMPARISON (Blurred / Deblurred / Sharp)\n')
    f.write('='*40 + '\n')
    f.write(summary.to_string())


In [ ]:
import os
import shutil
import random
import yaml
from ultralytics import YOLO
SOURCE_DIR = "./deblurred_results"    
DATASET_ROOT = "./deblur_dataset_RL"  
MODEL_VARIANT = "yolov8n.pt"           
TRAIN_SPLIT = 0.8                      
for sd in ['images/train', 'images/val', 'labels/train', 'labels/val']:
    os.makedirs(os.path.join(DATASET_ROOT, sd), exist_ok=True)

# 2. FILTER FOR RL IMAGES
# We use '_rl.png' because Task 2 showed it has superior SSIM (0.92)
images = [f for f in os.listdir(SOURCE_DIR) if f.endswith('_rl.png')]
random.seed(42)
random.shuffle(images)
split_point = int(len(images) * TRAIN_SPLIT)
labeler_model = YOLO(MODEL_VARIANT)
def build_dataset(file_list, stage):
    print(f" Processing {stage} set using Richardson-Lucy images...")
    for fname in file_list:
        img_src = os.path.join(SOURCE_DIR, fname)
        img_dst = os.path.join(DATASET_ROOT, 'images', stage, fname)
        shutil.copy(img_src, img_dst)
        results = labeler_model(img_src, verbose=False)
        label_path = os.path.join(DATASET_ROOT, 'labels', stage, fname.replace(".png", ".txt"))
        results[0].save_txt(label_path)
build_dataset(images[:split_point], 'train')
build_dataset(images[split_point:], 'val')
yaml_data = {
    'path': os.path.abspath(DATASET_ROOT),
    'train': 'images/train', 'val': 'images/val',
    'names': labeler_model.names
}
with open(os.path.join(DATASET_ROOT, 'dataset.yaml'), 'w') as f:
    yaml.dump(yaml_data, f)
print("\n Fine-tuning YOLO on RL-Restored Data...")
model = YOLO(MODEL_VARIANT)
model.train(data=os.path.join(DATASET_ROOT, 'dataset.yaml'), epochs=30, name="RL_deblur_detector")

In [ ]:
#task 5 Running the object dectection on the debulred data set of fineturning
Img= r"C:\Users\vcsd2\OneDrive\Desktop\Computer_vision\blur\images"
Results = "batch_detection_results.csv"
model = YOLO("yolov8n.pt")
batch_stats = []
image_files = [f for f in os.listdir(Img) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
print(f"🚀 Running Task 3 Detection on {len(image_files)} images...")
for filename in tqdm(image_files):
    img_path = os.path.join(Img, filename)
    results = model(img_path, verbose=False)
    for result in results:
        num_objects = len(result.boxes)
        avg_conf = result.boxes.conf.mean().item() if num_objects > 0 else 0
        inference_time = result.speed['inference']
        
        batch_stats.append({
            "filename": filename,
            "object_count": num_objects,
            "avg_confidence": avg_conf,
            "latency_ms": inference_time
        })
# 3. Save for report
df_task3 = pd.DataFrame(batch_stats)
df_task3.to_csv(Results, index=False)
print(f" Batch analysis complete, Data saved to {Results}")